In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

In [3]:
df = pd.read_csv("logistics_dataset.csv")


In [4]:
df


,item_id,category,stock_level,reorder_point,reorder_frequency_days,lead_time_days,daily_demand,demand_std_dev,item_popularity_score,storage_location_id,...,unit_price,holding_cost_per_unit_day,stockout_count_last_month,order_fulfillment_rate,total_orders_last_month,turnover_ratio,layout_efficiency_score,last_restock_date,forecasted_demand_next_7d,KPI_score
0,ITM10000,Pharma,283,21,4,4,49.85,1.56,0.43,L82,...,117.80,1.14,0,0.80,700,3.33,0.33,2024-02-17,184.37,0.556
1,ITM10001,Automotive,301,52,9,6,23.34,2.55,0.69,L15,...,178.80,1.09,3,0.79,736,10.36,0.98,2024-10-01,221.94,0.723
2,ITM10002,Groceries,132,60,11,8,37.69,3.15,0.62,L4,...,54.05,0.95,7,0.75,814,14.32,0.87,2024-04-07,53.85,0.680
3,ITM10003,Automotive,346,46,13,5,33.69,2.79,0.21,L95,...,31.10,1.90,0,0.96,994,2.08,0.29,2024-01-27,92.04,0.488
4,ITM10004,Automotive,49,55,4,6,49.58,5.23,0.31,L36,...,104.97,0.63,5,0.83,299,5.65,0.96,2024-05-17,194.58,0.670
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199,ITM13199,Groceries,343,21,12,2,39.88,1.30,0.34,L43,...,24.63,1.50,3,0.79,87,11.80,0.20,2024-11-28,237.04,0.545
3200,ITM13200,Electronics,428,43,5,7,2.68,4.25,0.91,L83,...,79.04,1.46,4,0.77,833,14.96,0.39,2024-11-28,34.00,0.605
3201,ITM13201,Groceries,415,80,14,5,49.15,5.41,0.14,L11,...,199.89,1.11,9,0.89,937,7.63,0.60,2024-10-02,62.57,0.509
3202,ITM13202,Groceries,173,84,3,9,43.39,8.47,0.69,L58,...,65.45,1.04,4,0.86,905,6.37,0.46,2024-03-30,36.96,0.565


In [5]:
cols_to_drop = [
    "storage_location_id",
    "zone",
    "picking_time_seconds",
    "handling_cost_per_unit",
    "layout_efficiency_score",
    "KPI_score",
]

df.drop(columns=cols_to_drop, inplace=True, errors="ignore")


In [6]:
df


,item_id,category,stock_level,reorder_point,reorder_frequency_days,lead_time_days,daily_demand,demand_std_dev,item_popularity_score,unit_price,holding_cost_per_unit_day,stockout_count_last_month,order_fulfillment_rate,total_orders_last_month,turnover_ratio,last_restock_date,forecasted_demand_next_7d
0,ITM10000,Pharma,283,21,4,4,49.85,1.56,0.43,117.80,1.14,0,0.80,700,3.33,2024-02-17,184.37
1,ITM10001,Automotive,301,52,9,6,23.34,2.55,0.69,178.80,1.09,3,0.79,736,10.36,2024-10-01,221.94
2,ITM10002,Groceries,132,60,11,8,37.69,3.15,0.62,54.05,0.95,7,0.75,814,14.32,2024-04-07,53.85
3,ITM10003,Automotive,346,46,13,5,33.69,2.79,0.21,31.10,1.90,0,0.96,994,2.08,2024-01-27,92.04
4,ITM10004,Automotive,49,55,4,6,49.58,5.23,0.31,104.97,0.63,5,0.83,299,5.65,2024-05-17,194.58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199,ITM13199,Groceries,343,21,12,2,39.88,1.30,0.34,24.63,1.50,3,0.79,87,11.80,2024-11-28,237.04
3200,ITM13200,Electronics,428,43,5,7,2.68,4.25,0.91,79.04,1.46,4,0.77,833,14.96,2024-11-28,34.00
3201,ITM13201,Groceries,415,80,14,5,49.15,5.41,0.14,199.89,1.11,9,0.89,937,7.63,2024-10-02,62.57
3202,ITM13202,Groceries,173,84,3,9,43.39,8.47,0.69,65.45,1.04,4,0.86,905,6.37,2024-03-30,36.96


In [7]:
df["last_restock_date"] = pd.to_datetime(df["last_restock_date"])


In [8]:
expiry_map = {
    "Pharma": 365,
    "Groceries": 30,
    "Automotive": 730,
    "Electronics": 540
}

df["expiry_days"] = df["category"].map(expiry_map)
df["expiry_date"] = df["last_restock_date"] + pd.to_timedelta(df["expiry_days"], unit="D")
df["days_to_expiry"] = (df["expiry_date"] - pd.Timestamp.today()).dt.days


In [9]:
df

,item_id,category,stock_level,reorder_point,reorder_frequency_days,lead_time_days,daily_demand,demand_std_dev,item_popularity_score,unit_price,holding_cost_per_unit_day,stockout_count_last_month,order_fulfillment_rate,total_orders_last_month,turnover_ratio,last_restock_date,forecasted_demand_next_7d,expiry_days,expiry_date,days_to_expiry
0,ITM10000,Pharma,283,21,4,4,49.85,1.56,0.43,117.80,1.14,0,0.80,700,3.33,2024-02-17,184.37,365.0,2025-02-16,-326.0
1,ITM10001,Automotive,301,52,9,6,23.34,2.55,0.69,178.80,1.09,3,0.79,736,10.36,2024-10-01,221.94,730.0,2026-10-01,266.0
2,ITM10002,Groceries,132,60,11,8,37.69,3.15,0.62,54.05,0.95,7,0.75,814,14.32,2024-04-07,53.85,30.0,2024-05-07,-611.0
3,ITM10003,Automotive,346,46,13,5,33.69,2.79,0.21,31.10,1.90,0,0.96,994,2.08,2024-01-27,92.04,730.0,2026-01-26,18.0
4,ITM10004,Automotive,49,55,4,6,49.58,5.23,0.31,104.97,0.63,5,0.83,299,5.65,2024-05-17,194.58,730.0,2026-05-17,129.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199,ITM13199,Groceries,343,21,12,2,39.88,1.30,0.34,24.63,1.50,3,0.79,87,11.80,2024-11-28,237.04,30.0,2024-12-28,-376.0
3200,ITM13200,Electronics,428,43,5,7,2.68,4.25,0.91,79.04,1.46,4,0.77,833,14.96,2024-11-28,34.00,540.0,2026-05-22,134.0
3201,ITM13201,Groceries,415,80,14,5,49.15,5.41,0.14,199.89,1.11,9,0.89,937,7.63,2024-10-02,62.57,30.0,2024-11-01,-433.0
3202,ITM13202,Groceries,173,84,3,9,43.39,8.47,0.69,65.45,1.04,4,0.86,905,6.37,2024-03-30,36.96,30.0,2024-04-29,-619.0


In [10]:
df["days_of_stock_left"] = df["stock_level"] / (df["daily_demand"] + 1e-5)
df["stockout_risk"] = (df["days_of_stock_left"] < df["lead_time_days"]).astype(int)

df["days_since_restock"] = (
    pd.Timestamp.today() - df["last_restock_date"]
).dt.days


In [11]:
df


,item_id,category,stock_level,reorder_point,reorder_frequency_days,lead_time_days,daily_demand,demand_std_dev,item_popularity_score,unit_price,...,total_orders_last_month,turnover_ratio,last_restock_date,forecasted_demand_next_7d,expiry_days,expiry_date,days_to_expiry,days_of_stock_left,stockout_risk,days_since_restock
0,ITM10000,Pharma,283,21,4,4,49.85,1.56,0.43,117.80,...,700,3.33,2024-02-17,184.37,365.0,2025-02-16,-326.0,5.677030,0,690
1,ITM10001,Automotive,301,52,9,6,23.34,2.55,0.69,178.80,...,736,10.36,2024-10-01,221.94,730.0,2026-10-01,266.0,12.896310,0,463
2,ITM10002,Groceries,132,60,11,8,37.69,3.15,0.62,54.05,...,814,14.32,2024-04-07,53.85,30.0,2024-05-07,-611.0,3.502254,1,640
3,ITM10003,Automotive,346,46,13,5,33.69,2.79,0.21,31.10,...,994,2.08,2024-01-27,92.04,730.0,2026-01-26,18.0,10.270107,0,711
4,ITM10004,Automotive,49,55,4,6,49.58,5.23,0.31,104.97,...,299,5.65,2024-05-17,194.58,730.0,2026-05-17,129.0,0.988302,1,600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199,ITM13199,Groceries,343,21,12,2,39.88,1.30,0.34,24.63,...,87,11.80,2024-11-28,237.04,30.0,2024-12-28,-376.0,8.600800,0,405
3200,ITM13200,Electronics,428,43,5,7,2.68,4.25,0.91,79.04,...,833,14.96,2024-11-28,34.00,540.0,2026-05-22,134.0,159.700897,0,405
3201,ITM13201,Groceries,415,80,14,5,49.15,5.41,0.14,199.89,...,937,7.63,2024-10-02,62.57,30.0,2024-11-01,-433.0,8.443538,0,462
3202,ITM13202,Groceries,173,84,3,9,43.39,8.47,0.69,65.45,...,905,6.37,2024-03-30,36.96,30.0,2024-04-29,-619.0,3.987093,1,648


In [12]:
y = df["forecasted_demand_next_7d"]
y

0       184.37
1       221.94
2        53.85
3        92.04
4       194.58
         ...  
3199    237.04
3200     34.00
3201     62.57
3202     36.96
3203    193.91
Name: forecasted_demand_next_7d, Length: 3204, dtype: float64

In [13]:
le = LabelEncoder()
df["category_encoded"] = le.fit_transform(df["category"])

In [14]:
df["is_expired"] = (df["days_to_expiry"] < 0).astype(int)

# Clip negative expiry days
df["days_to_expiry"] = df["days_to_expiry"].clip(lower=0)


In [15]:
features = [
    "category_encoded",
    "daily_demand",
    "demand_std_dev",
    "stock_level",
    "days_of_stock_left",
    "stockout_risk",
    "lead_time_days",
    "reorder_point",
    "reorder_frequency_days",
    "item_popularity_score",
    "total_orders_last_month",
    "stockout_count_last_month",
    "turnover_ratio",
    "days_since_restock",
    "days_to_expiry",
    "is_expired",
    "unit_price",
    "holding_cost_per_unit_day"
]

X = df[features]


In [16]:
X
y

0       184.37
1       221.94
2        53.85
3        92.04
4       194.58
         ...  
3199    237.04
3200     34.00
3201     62.57
3202     36.96
3203    193.91
Name: forecasted_demand_next_7d, Length: 3204, dtype: float64

In [17]:
X

,category_encoded,daily_demand,demand_std_dev,stock_level,days_of_stock_left,stockout_risk,lead_time_days,reorder_point,reorder_frequency_days,item_popularity_score,total_orders_last_month,stockout_count_last_month,turnover_ratio,days_since_restock,days_to_expiry,is_expired,unit_price,holding_cost_per_unit_day
0,4,49.85,1.56,283,5.677030,0,4,21,4,0.43,700,0,3.33,690,0.0,1,117.80,1.14
1,1,23.34,2.55,301,12.896310,0,6,52,9,0.69,736,3,10.36,463,266.0,0,178.80,1.09
2,3,37.69,3.15,132,3.502254,1,8,60,11,0.62,814,7,14.32,640,0.0,1,54.05,0.95
3,1,33.69,2.79,346,10.270107,0,5,46,13,0.21,994,0,2.08,711,18.0,0,31.10,1.90
4,1,49.58,5.23,49,0.988302,1,6,55,4,0.31,299,5,5.65,600,129.0,0,104.97,0.63
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199,3,39.88,1.30,343,8.600800,0,2,21,12,0.34,87,3,11.80,405,0.0,1,24.63,1.50
3200,2,2.68,4.25,428,159.700897,0,7,43,5,0.91,833,4,14.96,405,134.0,0,79.04,1.46
3201,3,49.15,5.41,415,8.443538,0,5,80,14,0.14,937,9,7.63,462,0.0,1,199.89,1.11
3202,3,43.39,8.47,173,3.987093,1,9,84,3,0.69,905,4,6.37,648,0.0,1,65.45,1.04


In [18]:
X = X.fillna(X.median())


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)


In [20]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)


XGBoostError: value -1 for Parameter max_depth should be greater equal to 0
max_depth: Maximum depth of the tree; 0 indicates no limit; a limit is required for depthwise policy

In [ ]:
!pip install xgboost


In [21]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

preds = model.predict(X_test)
preds = np.maximum(preds, 0)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print("MAE :", mae)
print("RMSE:", rmse)


MAE : 75.03643103636743
RMSE: 88.56044729017191


In [23]:
y_test.mean()

151.4189391575663

In [24]:
mae/y_test.mean()

0.4955551231163008

In [25]:
preds


array([111.60334 , 130.23898 , 172.25925 , 174.82486 , 159.67546 ,
       147.08235 , 156.14581 , 157.44844 , 194.44019 , 103.49948 ,
       151.25604 , 187.42311 , 141.88747 , 111.92925 , 157.07405 ,
       172.11006 , 170.22433 , 134.26805 , 186.37796 , 161.86005 ,
       164.46129 , 141.13168 , 223.61224 , 202.96193 , 173.64217 ,
       113.20464 , 163.76678 , 172.85655 , 103.0883  , 170.74246 ,
       146.5871  , 157.67484 , 136.9333  , 177.92366 , 170.25124 ,
       121.73778 , 130.98787 , 170.3499  , 146.6501  , 197.00975 ,
       150.89496 , 178.42546 , 144.56659 , 130.17711 , 162.04721 ,
       154.84094 , 160.4573  , 154.47096 , 153.60574 , 142.45772 ,
        92.67128 , 138.61543 , 132.99748 , 143.55154 , 148.07368 ,
       140.40869 , 149.63393 , 204.22455 , 134.00208 , 149.11397 ,
       106.28466 , 155.22227 , 132.78583 , 175.10005 , 186.53293 ,
       153.59352 , 174.90324 , 167.00168 , 193.46616 , 133.30382 ,
       141.2221  , 147.14572 , 138.89339 , 168.56238 , 126.050

In [26]:
y_test

2563    131.98
2564     68.65
2565    180.19
2566    121.97
2567     67.41
         ...  
3199    237.04
3200     34.00
3201     62.57
3202     36.96
3203    193.91
Name: forecasted_demand_next_7d, Length: 641, dtype: float64

In [28]:
import joblib

joblib.dump(model, "warehouse_demand_model.pkl")
joblib.dump(le, "category_encoder.pkl")   # save encoder too


['category_encoder.pkl']